# genulens Python API examples

This notebook demonstrates the direct Python API for `genulens`. The Python API calls the C++ simulation core and returns a typed `SimulationResult`; it does not parse the stdout of the `./genulens` command-line program.

Build the extension before opening the notebook:

```bash
cmake --build build --target genulens_python
PYTHONPATH=build jupyter notebook examples/python_binding.ipynb
```

If you installed the package with `pip install -e .`, setting `PYTHONPATH=build` is not needed.


In [ ]:
from pathlib import Path
import sys

# Find build/ whether Jupyter is launched from the repository root or examples/.
repo_root = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
build_dir = repo_root / "build"
if build_dir.exists():
    sys.path.insert(0, str(build_dir))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import genulens

plt.rcParams["figure.figsize"] = (6, 4)
print("genulens imported from:", genulens.__file__)


## Helper functions

`SimulationResult` carries column names and can be converted to a NumPy array. For analysis, a pandas DataFrame is usually the most convenient representation.


In [ ]:
def as_dataframe(result):
    return pd.DataFrame(result.to_numpy(), columns=result.columns)


def weighted_quantile(values, weights, q):
    values = np.asarray(values)
    weights = np.asarray(weights)
    order = np.argsort(values)
    cumulative = np.cumsum(weights[order])
    cumulative /= cumulative[-1]
    return values[order][np.searchsorted(cumulative, q)]


def summarize_events(df, label="sample"):
    w = df["weight"].to_numpy()
    summary = {
        "n": len(df),
        "weight_sum": w.sum(),
        "tE_p16": weighted_quantile(df["tE"], w, 0.16),
        "tE_p50": weighted_quantile(df["tE"], w, 0.50),
        "tE_p84": weighted_quantile(df["tE"], w, 0.84),
        "mass_p50": weighted_quantile(df["lens_mass_msun"], w, 0.50),
        "thetaE_p50": weighted_quantile(df["thetaE"], w, 0.50),
    }
    print(label)
    for key, value in summary.items():
        print(f"  {key:12s}: {value:.6g}" if isinstance(value, float) else f"  {key:12s}: {value}")
    return summary


## 1. Plain sampling

The minimal workflow is to create `Config(l, b, n_simu, seed)` and call `genulens.simulate()`. `n_simu` is the target number of accepted events returned by the simulation.


In [ ]:
cfg = genulens.Config(l=1.0, b=-3.9, n_simu=20_000, seed=42)

result = genulens.simulate(cfg)
df = as_dataframe(result)

print(result.columns)
df.head()


In [ ]:
summarize_events(df, "plain sampling")

bins_tE = np.logspace(-1, 3, 60)
fig, ax = plt.subplots()
ax.hist(df["tE"], bins=bins_tE, weights=df["weight"], density=True, histtype="stepfilled", alpha=0.35)
ax.axvline(weighted_quantile(df["tE"], df["weight"], 0.50), color="black", lw=1, label="weighted median")
ax.set_xscale("log")
ax.set_xlabel(r"$t_E$ [days]")
ax.set_ylabel("weighted density")
ax.legend()
plt.show()


## 2. Built-in observation constraints through typed config

Observation constraints such as `tE` and `thetaE` are set directly on `cfg.observation`. These values are passed as typed C++ configuration fields, not as command-line option strings.


In [ ]:
cfg_obs = genulens.Config(l=1.0, b=-3.9, n_simu=20_000, seed=42)
cfg_obs.observation.tE_obs = 54.5
cfg_obs.observation.tE_err = 5.0
cfg_obs.observation.thetaE_obs = 0.55
cfg_obs.observation.thetaE_err = 0.15

result_obs = genulens.simulate(cfg_obs)
df_obs = as_dataframe(result_obs)

summarize_events(df_obs, "built-in observation likelihood")
df_obs.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(df["tE"], bins=bins_tE, weights=df["weight"], density=True, histtype="step", label="plain")
axes[0].hist(df_obs["tE"], bins=bins_tE, weights=df_obs["weight"], density=True, histtype="step", label="obs")
axes[0].axvline(cfg_obs.observation.tE_obs, color="black", lw=1, ls="--")
axes[0].set_xscale("log")
axes[0].set_xlabel(r"$t_E$ [days]")
axes[0].legend()

bins_theta = np.linspace(0, 3, 60)
axes[1].hist(df["thetaE"], bins=bins_theta, weights=df["weight"], density=True, histtype="step", label="plain")
axes[1].hist(df_obs["thetaE"], bins=bins_theta, weights=df_obs["weight"], density=True, histtype="step", label="obs")
axes[1].axvline(cfg_obs.observation.thetaE_obs, color="black", lw=1, ls="--")
axes[1].set_xlabel(r"$\theta_E$ [mas]")
axes[1].legend()
plt.tight_layout()
plt.show()


## 3. Model, source-selection, and sampling config

Common model parameters are exposed under `cfg.model.*`. Source-selection and extinction settings live under `cfg.source.*`, while Monte Carlo behavior is controlled by `cfg.sampling.*`.


In [ ]:
cfg_model = genulens.Config(l=0.5, b=-2.0, n_simu=10_000, seed=7)

# Source selection and extinction
cfg_model.source.i_min = 14.0
cfg_model.source.i_max = 21.0
cfg_model.source.ai_rc = 1.5
cfg_model.source.evi_rc = 1.2

# Model parameters
cfg_model.model.imf.alpha2 = -1.35
cfg_model.model.density.stellar_halo = 0
cfg_model.model.nsd.enabled = 0
cfg_model.model.kinematics.omega_p = 45.0

# Sampling behavior
cfg_model.sampling.binary = 1
cfg_model.sampling.remnant = 1

result_model = genulens.simulate(cfg_model)
df_model = as_dataframe(result_model)
summarize_events(df_model, "typed model/source config")


## 4. Custom likelihood with multiple terms

Pass a Python callable as `likelihood=` to evaluate additional likelihood terms inside the `EventSampler` loop. The callable receives an `Event` and returns a multiplicative weight.

A return value less than or equal to zero rejects the event. Very narrow hard cuts can make the simulation slow because the sampler keeps drawing until enough accepted events are produced. A practical pattern is to keep hard cuts broad and encode detailed constraints as smooth likelihood factors.


In [ ]:
def gaussian(x, mu, sigma):
    z = (x - mu) / sigma
    return math.exp(-0.5 * z * z)


def complex_likelihood(event):
    # Broad hard cuts. Keeping these too narrow can make the run slow.
    if event.tE < 5.0 or event.tE > 300.0:
        return 0.0
    if event.thetaE <= 0.0 or event.lens_distance_pc <= 0.0 or event.source_distance_pc <= 0.0:
        return 0.0
    if event.lens_distance_pc >= event.source_distance_pc:
        return 0.0

    # Smooth constraints on tE, thetaE, lens mass, and lens/source distance ratio.
    like = 1.0
    like *= gaussian(event.tE, 54.5, 8.0)
    like *= gaussian(event.thetaE, 0.55, 0.18)
    like *= gaussian(math.log10(event.lens_mass_msun), math.log10(0.35), 0.45)

    distance_ratio = event.lens_distance_pc / event.source_distance_pc
    like *= gaussian(distance_ratio, 0.55, 0.25)

    # Example population prior: mildly prefer disk lenses over bar lenses.
    if event.lens_component <= 7:
        like *= 1.3
    elif event.lens_component == 8:
        like *= 0.8

    return like

cfg_custom = genulens.Config(l=1.0, b=-3.9, n_simu=20_000, seed=123)
result_custom = genulens.simulate(cfg_custom, likelihood=complex_likelihood)
df_custom = as_dataframe(result_custom)

summarize_events(df_custom, "custom likelihood")
df_custom.head()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df["tE"], bins=bins_tE, weights=df["weight"], density=True, histtype="step", label="plain")
axes[0].hist(df_custom["tE"], bins=bins_tE, weights=df_custom["weight"], density=True, histtype="step", label="custom")
axes[0].set_xscale("log")
axes[0].set_xlabel(r"$t_E$ [days]")
axes[0].legend()

axes[1].hist(df["thetaE"], bins=bins_theta, weights=df["weight"], density=True, histtype="step", label="plain")
axes[1].hist(df_custom["thetaE"], bins=bins_theta, weights=df_custom["weight"], density=True, histtype="step", label="custom")
axes[1].set_xlabel(r"$\theta_E$ [mas]")
axes[1].legend()

bins_mass = np.logspace(-2, 1.3, 60)
axes[2].hist(df["lens_mass_msun"], bins=bins_mass, weights=df["weight"], density=True, histtype="step", label="plain")
axes[2].hist(df_custom["lens_mass_msun"], bins=bins_mass, weights=df_custom["weight"], density=True, histtype="step", label="custom")
axes[2].set_xscale("log")
axes[2].set_xlabel(r"$M_L$ [$M_\odot$]")
axes[2].legend()

plt.tight_layout()
plt.show()


## 5. Weighted contribution by lens component

`lens_component` and `source_component` are included in the result columns. Component-level weighted fractions are useful for checking which Galactic population is selected by a likelihood model.


In [ ]:
component_names = {
    0: "thin disk 0", 1: "thin disk 1", 2: "thin disk 2", 3: "thin disk 3",
    4: "thin disk 4", 5: "thin disk 5", 6: "thin disk 6",
    7: "thick disk", 8: "bar", 9: "NSD", 10: "stellar halo",
}


def component_fractions(frame):
    total = frame["weight"].sum()
    rows = []
    for component, name in component_names.items():
        mask = frame["lens_component"] == component
        weight = frame.loc[mask, "weight"].sum()
        if weight > 0:
            rows.append((component, name, weight / total))
    return pd.DataFrame(rows, columns=["component", "name", "weighted_fraction"])

frac_plain = component_fractions(df).rename(columns={"weighted_fraction": "plain"})
frac_custom = component_fractions(df_custom).rename(columns={"weighted_fraction": "custom"})
frac_plain.merge(frac_custom, on=["component", "name"], how="outer").fillna(0)


## 6. `ruc` shortcut

For quick experiments, `genulens.ruc(l, b, n_simu, seed)` runs a simulation without explicitly constructing a `Config`. Use `Config` for any workflow with observation, source, model, or sampling options.


In [ ]:
quick = genulens.ruc(l=0.0, b=-2.0, n_simu=2_000, seed=1)
quick_df = as_dataframe(quick)
quick_df[["weight", "tE", "thetaE", "lens_mass_msun", "mu_rel_masyr"]].describe()
